In [ ]:
!pip install pymupdf langchain-text-splitters langchain-openai langchain-community chromadb

In [1]:
import fitz
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings  # 替換成 OpenAI
from langchain_community.vectorstores import Chroma

# 1. Load PDF
doc = fitz.open("2025舞光LED21st(單頁水印可搜尋).pdf")
text = "".join([page.get_text() for page in doc])

# 2. Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_text(text)

api_key=os.environ.get("OPENAI_API_KEY")
# 3. Embedding Model (切換為 OpenAI)
# 預設會使用 text-embedding-3-small，這是目前性價比最高的模型
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=api_key)



# 4. 向量資料庫處理
persist_dir = "./test_openai_embedding_vector_db"

# 注意：如果你之前用 HuggingFace 建立過同一個資料夾，必須先刪除它
# 因為不同模型的維度（Dimensions）不同，無法混用
if os.path.exists(persist_dir):
    import shutil
    shutil.rmtree(persist_dir)
    print(f"已刪除舊的向量庫，準備以 OpenAI 模型重建")

vector_db = Chroma.from_texts(
    chunks, 
    embeddings, 
    persist_directory=persist_dir
)

print(f"成功！已使用 OpenAI 嵌入模型建立資料庫。總共切分了 {len(chunks)} 個區塊。")

/Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


成功！已使用 OpenAI 嵌入模型建立資料庫。總共切分了 441 個區塊。


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define the LLM
llm = ChatOpenAI(model_name="gpt-5.2",reasoning={"effort": "medium"}, temperature=0)

# 3. Create the Prompt
template = """
### 角色定位
你是一位專業的「舞光 (DanceLight)」LED 照明產品顧問。你的任務是根據提供的目錄內容，為客戶提供準確、專業且親切的產品建議。

### 檢索到的目錄內容 (Context)
{context}

### 任務指令
1. **分析需求**：判斷使用者是尋找「特定產品名稱」（如：邱比特軌道燈）還是針對「特定場所」的需求（如：浴室、客廳）並且根據使用場景來推薦產品。
2. **規格導向**：在推薦產品時，請務必從 context 中提取以下技術屬性（若有標示）：
   - **瓦數 (W)**
   - **色溫 (K)**：如暖白光 (3000K)、自然光 (4000K) 或白光 (6000K)。
   - **演色性 (Ra)**：特別提到 Ra > 90 的高演色性產品。
   - **特殊功能**：如 IP65 防水等級、防眩光、低頻閃等。
3. **排版方式**：請使用「列點」格式，讓產品規格一目了然。

### 回答限制
- **範疇限制**：僅根據上方提供的「檢索到的目錄內容」進行回答。如果 context 中沒有相關資訊，請禮貌地回答：「抱歉，目前的產品目錄中沒有提到相關細節，建議您可以詢問其他類似款式。」
- **嚴禁胡說**：不可自行編造產品型號、規格或價格。
- **語言一致性**：一律使用「繁體中文」回答。

---
使用者問題：{question}
顧問建議：
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build the Chain using LCEL (Modern 1.2.x style)
# This replaces RetrievalQA entirely
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask a question
# Note: Since it's a simple LCEL chain, you pass the string directly
response = qa_chain.invoke("可以放在浴室的燈有哪些？")
print(response)

以下依「浴室（潮濕空間）」需求，從目錄中挑選**具防水/耐潮濕訴求**的款式（以吸頂燈、壁燈為主）供您參考：

- **白鯨防水吸頂燈（戶外門廊/潮濕區域適用）**
  - 瓦數：**16W / 24W**
  - 色溫：**3000K（暖白光）**（目錄中 24W 為 3000K）
  - 光通量：**24W：2640 lm**
  - 演色性（Ra）：目錄未標示
  - 特殊/特色：
    - **一體成型**、輕巧好安裝
    - **抗 UV PP 材質**，不易泛黃脆化
    - **快裝結構**（按壓快裝/旋轉快拆）
    - 目錄明確寫「**適用戶外門廊等潮濕區域**」
  - 燈體顏色：目錄出現 **貴族黑**（型號：**E-CEBJ24W-BK**）

- **白鯨防水壁燈（IP66）**（適合浴室牆面、乾濕分離外側牆面等）
  - 防護等級：**IP66**
  - 發光角度：**150°**
  - 輸入電壓：**100–240V（全電壓）**
  - 燈體材質：**抗 UV PP**
  - 規格（可選 10W / 16W；3000K / 6500K）：
    - **10W / 3000K（暖白光）/ 1100 lm / Ra≧80**
      - 型號：**E-WLBJ10W**、**E-WLBJ10W-BK**
    - **10W / 6500K（白光）/ 1150 lm / Ra≧80**
      - 型號：**E-WLBJ10D**、**E-WLBJ10D-BK**
    - **16W / 3000K（暖白光）/ 1750 lm / Ra≧80**
      - 型號：**E-WLBJ16W**、**E-WLBJ16W-BK**
    - **16W / 6500K（白光）/ 1850 lm / Ra≧80**
      - 型號：**E-WLBJ16D**、**E-WLBJ16D-BK**
  - 安裝提醒：目錄有提到「**適用直式預埋盒**」（壁燈安裝更容易對位）

- **微波感應戶外吸頂燈（若您希望“人來自動亮”）**
  - 瓦數：**16W**
  - 色溫：**6250K（偏白光）**
  - 光通量：**1600 lm**
  - 演色性（Ra）：**≧80**
  - 輸入電壓：**

In [5]:
import sys
import langchain
print(f"Python Path: {sys.executable}")
print(f"LangChain Version: {langchain.__version__}")
print(f"LangChain Location: {langchain.__file__}")

Python Path: /Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/bin/python
LangChain Version: 1.2.10
LangChain Location: /Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/lib/python3.11/site-packages/langchain/__init__.py
